# What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

## Without a Gateway
- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query

## With a Gateway
- One unified API for 100+ providers
- Automatic fallbacks if a provider fails
- Centralized logging, cost tracking, rate limiting
- Swap models with a config change, no code rewrite
- Cache repeated queries → save money

## Installation & Setup
- LiteLLM → the most popular open-source LLM gateway (supports 100+ providers)
- LangChain → for building agentic workflows on top of the gateway
- python-dotenv → for managing API keys

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from litellm import completion
import litellm
litellm.suppress_debug_info = True

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

# The Simplest LiteLLM Example — Unified API

The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function — completion() — that works with all of them. Look at how clean this is:

In [5]:
from litellm import completion

# Same code, different providers — just change the `model` string!

response_google = completion(
    model="gemini/gemini-2.5-flash-lite",
    messages=[{"role":"user", "content":"Explain RAG in one sentence."}]
)
print(f"GEMINI: ", response_google.choices[0].message.content)

response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role":"user", "content":"Explain RAG in one sentence."}]
)

print(f"GROQ: ", response_groq.choices[0].message.content)



GEMINI:  RAG (Retrieval Augmented Generation) is a technique that enhances language models by first retrieving relevant information from an external knowledge base and then using that information to generate a more accurate and informed response.
GROQ:  RAG (Retrieve, Augment, Generate) is an artificial intelligence model that combines retrieval and generation capabilities to provide more accurate and informative responses by retrieving relevant information from a database or knowledge graph and then using that information to generate a response.


In [6]:
from litellm import completion

prompt = "Explain RAG in one sentence"

providers = [
    ("OpenAI", "gpt-4o-mini"),
    ("Groq", "groq/llama-3.3-70b-versatile"),
    ("Anthropic", "claude-3-5-haiku-20241022"),
    ("Gemini", "gemini/gemini-2.5-flash-lite")
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r=completion(model=model, messages=[{"role":"user", "content":prompt}])
        print(f"{label}: {r.choices[0].message.content}")
    except Exception as e:
        print(f"{label}: {type(e).__name__}")

OpenAI: RateLimitError
Groq: RAG (Retrieve, Augment, Generate) is a type of AI model that combines retrieval of relevant information from a database or knowledge base with generation capabilities to produce more accurate and informative responses.
Anthropic: BadRequestError
Gemini: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by first retrieving relevant information from an external knowledge source and then using that information to generate a more informed and accurate response.


# Automatic Fallbacks — When OpenAI Goes Down
Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [7]:
from litellm import completion

# Define a fallback chain: try GPT first, then Groq and gemini
response = completion(
    model="gpt-4o-mini",
    messages=[{"role":"user", "content":"What is LLM Gateway?"}],
    fallbacks=[
        "groq/llama-3.3-70b-versatile",
        "gemini/gemini-2.5-flash-lite"
    ]
)
print("Response:", response.choices[0].message.content)
print("Actually answered model is:", response.model)

22:20:55 - LiteLLM:ERROR: fallback_utils.py:68 - Fallback attempt failed for model gpt-4o-mini: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Traceback (most recent call last):
  File "d:\projects\LLM Gateway\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 930, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "d:\projects\LLM Gateway\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\LLM Gateway\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 461, in make_openai_chat_c

Response: LLM Gateway is an interface or platform that allows users to interact with Large Language Models (LLMs) in a more user-friendly and accessible way. The term "gateway" suggests a portal or an entry point that connects users to the capabilities of LLMs, which can be complex and require technical expertise to interact with directly.

An LLM Gateway typically provides features such as:

1. **Text-based interface**: Allowing users to input text prompts or questions and receive responses from the LLM.
2. **Simplified input**: Enabling users to provide input in a more straightforward and intuitive way, without requiring extensive technical knowledge.
3. **Output formatting**: Presenting the output from the LLM in a readable and understandable format.
4. **Configuration options**: Allowing users to customize the behavior of the LLM, such as selecting the model, setting parameters, or choosing the output format.

The primary purpose of an LLM Gateway is to make LLMs more accessible to

If gpt-4o-mini is rate-limited or down, LiteLLM transparently retries with Groq, then Gemini. Your app never sees the failure.

This is the #1 reason teams adopt an LLM Gateway.